In [5]:
import numpy as np
import pandas as pd
from scipy.optimize import nnls

df = pd.read_csv("data.csv", index_col=0)
df

,PSC,JUNTS,ERC,PP,VOX,COMUNS,CUP,ALIANÇA.CAT,Altres candidatures,Cs,Abstenció
1980,0.138185,0.171275,0.054873,0.014556,0.000000,0.115557,0.000000,0.000000,0.117144,0.000000,0.388410
1984,0.193811,0.301318,0.028411,0.049647,0.000000,0.035962,0.000000,0.000000,0.031802,0.000000,0.359048
1988,0.177197,0.272179,0.024616,0.031648,0.000000,0.046166,0.000000,0.000000,0.039541,0.000000,0.408652
1992,0.152089,0.255296,0.043962,0.032968,0.000000,0.035913,0.000000,0.000000,0.025621,0.000000,0.454152
1995,0.159822,0.263467,0.061096,0.084246,0.000000,0.062607,0.000000,0.000000,0.005756,0.000000,0.363006
1999,0.227821,0.226843,0.052261,0.057211,0.000000,0.015129,0.000000,0.000000,0.016845,0.000000,0.403890
2003,0.198586,0.196993,0.104874,0.075627,0.000000,0.046480,0.000000,0.000000,0.008517,0.000000,0.368922
2006,0.153722,0.180767,0.080571,0.060980,0.000000,0.054760,0.000000,0.000000,0.013357,0.017425,0.438418
2010,0.111491,0.234199,0.042640,0.075148,0.000000,0.044927,0.000000,0.000000,0.061473,0.020696,0.409426
2012,0.101217,0.215104,0.095984,0.091014,0.000000,0.069380,0.024398,0.000000,0.040548,0.053099,0.309256


Iniial matriu transició

In [6]:
# 1. Dades (eliminem la primera fila de 1980 com has demanat)
# Ara X començarà a partir de 1984
# 2. Preparar les matrius
# Saltem la primera fila (1980) amb .iloc[1:]

dades = df.iloc[1:].values  # Matriu de dades numèriques
df = df.iloc[-11:]
partits = df.columns        # Noms dels partits

# X_t ≈ P * X_{t+1}
# A seran els resultats "futurs"
# B seran els resultats "passats"
A = dades[1:]  # De la segona fila fins al final
B = dades[:-1] # De la primera fila fins a la penúltima

P_llista = []

# 3. Resolem amb la condició de NO-NEGATIVITAT (nnls)
for j in range(len(partits)):
    # nnls retorna (vector_solucio, residu)
    p_j, _ = nnls(A, B[:, j])
    P_llista.append(p_j)

# 4. Normalització i formatat
P_est = np.array(P_llista).T

# Opcional: Normalitzem les columnes perquè sumin 1 (perquè sigui una probabilitat real)
P_final = P_est / (P_est.sum(axis=0) + 1e-12)

df_P = pd.DataFrame(P_final, index=partits, columns=partits)

print("Matriu de Transició (1984-2024) - Només valors positius:")
print(df_P.round(3).to_string())

Matriu de Transició (1984-2024) - Només valors positius:
                       PSC  JUNTS    ERC     PP  VOX  COMUNS    CUP  ALIANÇA.CAT  Altres candidatures     Cs  Abstenció
PSC                  0.552  0.172  0.000  0.154  0.0   0.040  0.000          0.0                0.000  0.000      0.000
JUNTS                0.188  0.492  0.000  0.000  0.0   0.008  0.000          0.0                0.042  0.000      0.074
ERC                  0.000  0.000  0.101  0.000  0.0   0.000  0.000          0.0                0.000  0.002      0.000
PP                   0.000  0.000  0.000  0.276  0.0   0.000  0.000          0.0                0.000  0.000      0.157
VOX                  0.000  0.000  0.713  0.000  0.0   0.000  0.658          0.0                0.000  0.797      0.000
COMUNS               0.181  0.000  0.000  0.000  0.0   0.000  0.000          0.0                0.405  0.000      0.000
CUP                  0.000  0.336  0.000  0.000  0.0   0.187  0.000          0.0                0.271  

Eliminem aliança i altres partits

In [7]:
# 1. Dades (eliminem la primera fila de 1980 com has demanat)
# Ara X començarà a partir de 1984
# 2. Preparar les matrius

#eliminem columnes
df = df.drop(columns=['ALIANÇA.CAT', 'Altres candidatures'])

df = df.div(df.sum(axis=1), axis=0)

# Saltem la primera fila (1980) amb .iloc[1:]
dades = df.iloc[1:].values  # Matriu de dades numèriques
# Per a la visualització, només agafem les últimes 8 files (1984-2024)
df = df.iloc[-9:]
partits = df.columns        # Noms dels partits

# X_t ≈ P * X_{t+1}
# A seran els resultats "futurs"
# B seran els resultats "passats"
A = dades[1:]  # De la segona fila fins al final
B = dades[:-1] # De la primera fila fins a la penúltima

P_llista = []

# 3. Resolem amb la condició de NO-NEGATIVITAT (nnls)
for j in range(len(partits)):
    # nnls retorna (vector_solucio, residu)
    p_j, _ = nnls(A, B[:, j])
    P_llista.append(p_j)

# 4. Normalització i formatat
P_est = np.array(P_llista).T

# Opcional: Normalitzem les columnes perquè sumin 1 (perquè sigui una probabilitat real)
P_final = P_est / (P_est.sum(axis=0) + 1e-12)

df_P = pd.DataFrame(P_final, index=partits, columns=partits)

print("Matriu de Transició (1984-2024) - Només valors positius:")
print(df_P.round(3).to_string())

Matriu de Transició (1984-2024) - Només valors positius:
             PSC  JUNTS    ERC     PP    VOX  COMUNS    CUP     Cs  Abstenció
PSC        0.444  0.174  0.000  0.000  0.000   0.000  0.000  0.000      0.067
JUNTS      0.000  0.404  0.000  0.561  0.000   0.136  0.000  0.000      0.014
ERC        0.000  0.095  0.000  0.000  0.000   0.000  0.000  0.002      0.000
PP         0.260  0.000  0.000  0.000  0.041   0.000  0.000  0.000      0.557
VOX        0.000  0.000  0.683  0.000  0.959   0.000  0.743  0.790      0.347
COMUNS     0.274  0.000  0.000  0.000  0.000   0.000  0.000  0.000      0.000
CUP        0.000  0.327  0.000  0.281  0.000   0.406  0.000  0.000      0.000
Cs         0.000  0.000  0.229  0.158  0.000   0.338  0.257  0.208      0.015
Abstenció  0.023  0.000  0.088  0.000  0.000   0.120  0.000  0.000      0.000


3.Posem limitacions per esquerres i dretes

In [8]:
from scipy.optimize import minimize

# 1. Definició de les prohibicions ideològiques
# (Partit d'origen: llista de partits on és impossible que vagi el vot)
prohibicions = {
    'CUP': ['VOX', 'PP', 'Cs'],
    'VOX': ['CUP', 'ERC', 'COMUNS'],
    'PP': ['CUP', 'ERC'],
    'ERC': ['VOX', 'PP'],
    'JUNTS': ['VOX', 'PP'],
    'PSC': ['CUP'],
    'Cs': ['CUP', 'ERC']
}

# 2. Preparació de les dades (assegura't que df_norm és el teu DF normalitzat)
# Si no el tens normalitzat, descomenta la següent línia:
# df_norm = df.div(df.sum(axis=1), axis=0)

dades = df.iloc[1:].values 
partits = df.columns
n = len(partits)

# A: Futur (X_t+1), B: Passat (X_t)
A = dades[1:]  
B = dades[:-1] 

P_est = np.zeros((n, n))

# 3. Bucle d'optimització amb restriccions
for j in range(n):
    # b és la sèrie històrica del partit d'origen j que volem explicar
    b = B[:, j]
    partit_origen = partits[j]
    
    def objectiu(p):
        # Error de mínims quadrats (predicció vs realitat)
        error = np.sum((A @ p - b) ** 2)
        
        # PENALITZACIÓ: Afegim un cost a qualsevol valor que no sigui la diagonal
        # Això força al model a mantenir la gent al mateix partit si és possible.
        # El coeficient 0.05 es pot pujar per forçar més fidelitat.
        cost_transvase = 0.05 * np.sum(p[np.arange(len(p)) != j]**2)
        
        return error + cost_transvase

    # Restriccions: La suma de la columna ha de ser 1
    cons = [{'type': 'eq', 'fun': lambda p: np.sum(p) - 1}]
    
    # Límits per defecte (probabilitats entre 0 i 1)
    bounds = [(0, 1)] * n
    
    # APLICAR PROHIBICIONS
    if partit_origen in prohibicions:
        for partit_desti in prohibicions[partit_origen]:
            if partit_desti in partits:
                idx_desti = list(partits).index(partit_desti)
                bounds[idx_desti] = (0, 0) # Forcem probabilitat zero estricta

    # Punt de partida: Fidelitat total (tothom es queda on està)
    p0 = np.zeros(n)
    p0[j] = 1
    
    res = minimize(objectiu, p0, method='SLSQP', bounds=bounds, constraints=cons, 
                   options={'ftol': 1e-9, 'maxiter': 1000})
    
    P_est[:, j] = res.x

# 4. Resultat final
df_P_final = pd.DataFrame(P_est, index=partits, columns=partits)

print("### MATRIU DE TRANSICIÓ AMB RESTRICCIONS IDEOLÒGIQUES ###")
print(df_P_final.round(3).to_string())

### MATRIU DE TRANSICIÓ AMB RESTRICCIONS IDEOLÒGIQUES ###
             PSC  JUNTS    ERC     PP    VOX  COMUNS    CUP     Cs  Abstenció
PSC        0.845  0.000  0.066  0.000  0.000   0.001  0.002  0.004      0.000
JUNTS      0.087  0.777  0.000  0.017  0.000   0.000  0.000  0.000      0.232
ERC        0.000  0.079  0.696  0.000  0.000   0.016  0.002  0.000      0.000
PP         0.029  0.000  0.000  0.823  0.008   0.000  0.000  0.000      0.000
VOX        0.000  0.000  0.000  0.005  0.988   0.033  0.000  0.262      0.000
COMUNS     0.020  0.000  0.049  0.042  0.000   0.888  0.000  0.033      0.000
CUP        0.000  0.000  0.112  0.000  0.000   0.024  0.996  0.000      0.000
Cs         0.000  0.061  0.000  0.114  0.004   0.026  0.000  0.625      0.000
Abstenció  0.019  0.082  0.076  0.000  0.000   0.011  0.000  0.076      0.768


Trobar l'equilibri a llarg termini equival a resoldre $P\pi=\pi$, que és el mateix que trobar el vector propi $\pi$ de valor propi $\lambda=1$


In [16]:
import numpy as np

eigenvalues, eigenvectors = np.linalg.eig(df_P_final)

# Trobar l'índex de l'autovalor més proper a 1
idx = np.argmin(np.abs(eigenvalues - 1))

# Extreure el vector i normalitzar (suma = 1)
pi = np.real(eigenvectors[:, idx])
pi = pi / pi.sum()

partits = ['PSC','JUNTS','ERC','PP','VOX','COMUNS','CUP','Cs','Abstenció']
for p, v in zip(partits, pi):
    print(f"{p:10s}: {v:.4f}")

###### COMPROVACIÓ DE LA DISTRIBUCIÓ ESTACIONÀRIA ######
print("\nComprovació de la distribució estacionària:")
print(np.allclose(df_P_final @ pi, pi))  # Ha de donar True
print(pi.sum())                    # Ha de donar 1.0

PSC       : 0.0112
JUNTS     : 0.0278
ERC       : 0.0110
PP        : 0.0210
VOX       : 0.4287
COMUNS    : 0.0198
CUP       : 0.4423
Cs        : 0.0174
Abstenció : 0.0210

Comprovació de la distribució estacionària:
True
0.9999999999999999
